In [1]:
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
import pandas as pd

df = pd.read_csv('train.csv')
df['Age'] = df['Age'].fillna(df['Age'].median())
df = df.drop('Cabin', axis=1)
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

X = df[['Pclass', 'Age', 'Fare', 'SibSp', 'Parch']]
y = df['Survived']


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Bug 1 — CV on test data only
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
scores = cross_val_score(dt, X, y, cv=5)
print("CV scores:", scores.mean())

# Bug 2 — missing std check
rf = RandomForestClassifier(n_estimators=100, random_state=42)
scores = cross_val_score(rf, X, y, cv=5)
print("RF CV score:", scores.mean())
print("RF CV STD",scores.std())

# Bug 3 — GridSearch on wrong data
param_grid = {'max_depth': [3, 5, 7]}
gs = GridSearchCV(DecisionTreeClassifier(random_state=42),
                  param_grid, cv=5)
gs.fit(X_train, y_train)
print("Best params:", gs.best_params_)

# Bug 4
param_grid = {'max_depth': [3, 5, 7],
              'n_estimators': [50, 100]}
gs = GridSearchCV(RandomForestClassifier(random_state=42),
                  param_grid, cv=5)
gs.fit(X_train, y_train)
print("Best score:", gs.best_score_)
print('BEST PARAMS',gs.best_params_)

# Bug 5
best_model = gs.best_estimator_
print("Test accuracy:", best_model.score(X_test,y_test))

CV scores: 0.7037913501977278
RF CV score: 0.6779235452890591
RF CV STD 0.019680120662291065
Best params: {'max_depth': 5}
Best score: 0.7274697133852064
BEST PARAMS {'max_depth': 3, 'n_estimators': 50}
Test accuracy: 0.7318435754189944
